In [ ]:
!pip install -q transformers datasets accelerate peft jiwer soundfile librosa evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.7 MB/s eta 0:00:00


In [ ]:
!pip install -U "torchao>=0.16.0"
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 35.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.0 MB/s eta 0:00:00


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import pandas as pd
import numpy as np
import librosa
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, LoraModel
import jiwer

from huggingface_hub import login
HF_TOKEN = "" #token erased for form submission
login(token=HF_TOKEN)

DEVICE = "cuda:0"
MODEL_NAME = "ixxan/whisper-small-common-voice-ug"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [ ]:
DATA_DIR = "/kaggle/input/competitions/dlp-26-t-2-nppe-2"
WAV_DIR  = os.path.join(DATA_DIR, "wavs")

train_csv_path = os.path.join(DATA_DIR, "train.csv")
test_csv_path  = os.path.join(DATA_DIR, "test.csv")

In [ ]:
train_df = pd.read_csv(train_csv_path)
test_df  = pd.read_csv(test_csv_path)

# Map filenames to full path
train_df['filepath'] = train_df['filepath'].apply(lambda x: os.path.join(WAV_DIR, os.path.basename(x)))
test_df['filepath']  = test_df['filepath'].apply(lambda x: os.path.join(WAV_DIR, os.path.basename(x)))

print(f"Loaded Train Samples: {len(train_df)}")
print(f"Loaded Test Samples:  {len(test_df)}")

# Filter extreme outliers (max ~63s vs 95th pctile ~19.5s)
import librosa as lb
train_df["duration"] = train_df["filepath"].apply(lambda f: lb.get_duration(path=f))
train_df = train_df[train_df["duration"] <= 20.0].reset_index(drop=True)
train_df = train_df.drop(columns=["duration"])

print(f"Train: {len(train_df)}")

# Initialize Processor & Tokenizer
# Omit language and task for unsupported languages like Uyghur ('ug')
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME)
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

# preprocessing function
def prepare_dataset(batch):
    audio, _ = librosa.load(batch["filepath"], sr=16000)
    batch["input_features"] = feature_extractor(audio, sampling_rate=16000).input_features[0]

    if "transcription" in batch and pd.notna(batch["transcription"]):
        batch["labels"] = tokenizer(batch["transcription"]).input_ids
    return batch

print("Extracting Mel-Spectrogram features...")
train_dataset = train_dataset.map(
    prepare_dataset,
    remove_columns=train_dataset.column_names
)
test_dataset = test_dataset.map(
    prepare_dataset,
    remove_columns=[c for c in test_dataset.column_names if c != "ID"]
)

Loaded Train Samples: 7574
Loaded Test Samples:  1894
Train: 7235


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Extracting Mel-Spectrogram features...


Map:   0%|          | 0/7235 [00:00<?, ? examples/s]

Map:   0%|          | 0/1894 [00:00<?, ? examples/s]

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        if "labels" in features[0]:
            label_features = [{"input_ids": feature["labels"]} for feature in features]
            labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
            labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

            if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
                labels = labels[:, 1:]

            batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    cer = jiwer.cer(label_str, pred_str)
    return {"cer": cer}

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# SpecAugment — helps generalization on small data
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05


peft_config = LoraConfig(
    r=32,                    # bumped from 16 — more capacity
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],  # wider than just q/v
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()
model.config.use_cache = False

model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 12,976,128 || all params: 254,711,040 || trainable%: 5.0945


In [ ]:
#Training model
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-uyghur-peft",
    per_device_train_batch_size= 16, #8,
    gradient_accumulation_steps=2, #4,    # Effective Batch Size = 32
    learning_rate= 1e-3, #5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    num_train_epochs=15,
    gradient_checkpointing= True,
    fp16=True,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none"
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting Fine-Tuning...")
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting Fine-Tuning...


Step,Training Loss
50,9.438046
100,2.527943
150,1.550376
200,1.242802
250,1.050838
300,0.937421
350,0.992272
400,0.933649
450,0.916973
500,0.721827


TrainOutput(global_step=3405, training_loss=0.4939299465582409, metrics={'train_runtime': 33026.6383, 'train_samples_per_second': 3.286, 'train_steps_per_second': 0.103, 'total_flos': 3.3346587949056e+19, 'train_loss': 0.4939299465582409, 'epoch': 15.0})

In [ ]:
#Inference on test data
print("Generating transcriptions for Test Dataset...")
model.eval()

predictions = []
ids = []
batch_size = 16

for i in range(0, len(test_df), batch_size):
    batch_df = test_df.iloc[i : i + batch_size]

    input_features_list = []
    for fp in batch_df['filepath']:
        audio, _ = librosa.load(fp, sr=16000)
        feat = feature_extractor(audio, sampling_rate=16000).input_features[0]
        input_features_list.append(feat)

    input_tensor = torch.tensor(np.array(input_features_list)).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            input_features=input_tensor,
            max_new_tokens=225,
            num_beams=5 #3
        )

    transcriptions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    predictions.extend(transcriptions)
    ids.extend(batch_df['ID'].tolist())

# Format output file
submission_df = pd.DataFrame({
    'ID': ids,
    'transcription': predictions
})

# Basic text cleanup
submission_df['transcription'] = submission_df['transcription'].str.strip()

submission_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_path, index=False)
print("File generated successfully..")

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Generating transcriptions for Test Dataset...


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `

File generated successfully..


In [ ]:
submission_df.head()

,ID,transcription
0,f068a206b84c4632865e0629a1b62fb8,bu dorini helila qaynatqan caqqan bol vissiqid...
1,a9d8cfab47b34f12b8f4b4769075713e,yamGurdin keyinki hawa Huddi sUzUp tazilanGand...
2,34147b4f995144288b720d7474ba4dd6,qar barGancA qattiq yaGdi yoldiki piyadilAr te...
3,c6c201bcd81a402385c2f008983f7474,cAtkA ciqip bilim vigAligAndin keyin qaytip ke...
4,c3c190cc67c14d4a946ef1b722196248,vAyiblAx kixini cUxkUnlAxtUridu vilhamlandurux...
